## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://private-user-images.githubusercontent.com/181020547/465824799-12a2371b-8be2-4219-9b48-90503eb43c69.png?jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbSIsImtleSI6ImtleTUiLCJleHAiOjE3NjAwNDgyMzcsIm5iZiI6MTc2MDA0NzkzNywicGF0aCI6Ii8xODEwMjA1NDcvNDY1ODI0Nzk5LTEyYTIzNzFiLThiZTItNDIxOS05YjQ4LTkwNTAzZWI0M2M2OS5wbmc_WC1BbXotQWxnb3JpdGhtPUFXUzQtSE1BQy1TSEEyNTYmWC1BbXotQ3JlZGVudGlhbD1BS0lBVkNPRFlMU0E1M1BRSzRaQSUyRjIwMjUxMDA5JTJGdXMtZWFzdC0xJTJGczMlMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI1MTAwOVQyMjEyMTdaJlgtQW16LUV4cGlyZXM9MzAwJlgtQW16LVNpZ25hdHVyZT1iYTRmYTAzYjkzYjA2MGE4ZTZlYjQ4ODU1OWIwY2VlZWU0Mzk0YzdmMjQ1YTlhMDMyNmI3NWNlZTQxNDdlZGViJlgtQW16LVNpZ25lZEhlYWRlcnM9aG9zdCJ9.a8477QD1J4Lrmys7jB8gt_H5pdiKBsKsu3npEqZjEpo)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

## Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 1: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

#### ❓ Question 1:

 Explain the interrelationships between the three states.  Why don't we just make a single huge state?


 ANSWER ✅

 There are three main states that are hierarchical in nature. The Agent State is the top level and serves as the entry point to the system. It tracks user requests, overall messages, notes, outputs, and the final report. It acts as the highest-level workflow orchestrator and delegates tasks to the next level down, the Supervisor State.

The Supervisor State is important because it plans, delegates, and monitors research tasks. This middle layer handles mid-level planning and serves as iteration and feedback control, as well as loop management, instead of sending everything back through the Agent State. It is a subgraph that manages the supervisor's messages, research iterations, and coordinates parallel researchers.

The last and bottom-level state is the Researcher State, which represents individual researchers. Each researcher maintains its own message history, tool call iterations, and coordinates parallel research tasks. In layman’s terms, the Agent State is the CEO (big goals, final checks, and reports), the Supervisor State is the Project Manager (plans, delegates subtasks, monitors progress, and iterates), and the Research State represents the SMEs or boots-on-the-ground workers (perform the actual data gathering, summarization, and tasks themselves). Having all these layers is important because it creates boundaries that help manage complexity more efficiently.

You do not want a single huge state. Each layer handles a different responsibility, and combining them all into one state would mix responsibilities, making the system more prone to errors and harder to troubleshoot. Without layers, workflows could not be reused independently, efficiency would drop, and logic might be duplicated across tasks. A single huge state is also not scalable. Multiple layers allow for asynchronous work and parallel execution of subunits. With only one state, you would have to track all partial results, workflows, tasks, and iterations, which would be computationally heavy, expensive, difficult to manage, and prone to disorganization. Multiple layers provide structure, maintainability, and efficiency.




 

## Task 2: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

### ❓ Question 2:  

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

ANSWER ✅

Importing these components is important because the utilities contain a lot of underlying code, which would heavily bloat the notebook if included directly. By using the code “under the hood” through imports instead of displaying it in the notebook, the notebook remains cleaner and easier to read. Importing also makes it faster to use the components, allows you to reuse the same tools across multiple notebooks, and ensures that any updates to the library are automatically available without manually changing source code. This approach improves scalability and makes notebook maintenance much easier.

The disadvantages are that the library must be present for the notebook to work; if it is missing, the notebook will fail. Versioning issues can also arise, so it’s important to maintain a configuration file (e.g., .toml) specifying the required package versions to help others run the code. Additionally, the imported code is less customizable, making it harder to modify if needed, and there is less transparency, which can make debugging more difficult.



## Task 3: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 4: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Task 5: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 6: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 7: Running the Deep Researcher

Now let's see the system in action! We'll use it to analyze a PDF document about how people use AI.

### Setup

We need to:
1. Load the PDF document
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Load the PDF document
from pathlib import Path
import PyPDF2

def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

# Load the PDF about how people use AI
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)

print(f"Loaded PDF with {len(pdf_content)} characters")
print(f"First 500 characters:\n{pdf_content[:500]}...")

Loaded PDF with 112460 characters
First 500 characters:
NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/w34255
NATIONAL BUREAU OF ECONOMIC RESEARCH
1050 Massachusetts Avenue
Cambridge, MA 02138
September 2025
We acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan 
Beiermeister,  Rachel Brown, Cassandra Duchan Solis, Jason Kwon, Elliott Mokski, Kevin Rao, 
Harrison Satcher,  Gawe...


In [16]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (3 concurrent researchers)
- **Clarification enabled** (will ask if research scope is unclear)

In [17]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researchers
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Search API: Tavily


### Execute the Research

Now let's run the research! We'll ask the system to analyze the PDF and provide insights about how people use AI.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Multiple researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [18]:
# Create our research request with PDF context
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with your analysis request. You've provided a comprehensive PDF document from the NBER Working Paper Series titled "How People Use ChatGPT" by Chatterji et al. (2025), and you've clearly outlined three specific areas for analysis:

1. Main findings about how people are using AI
2. Most common use cases 
3. Trends and patterns emerging from the data

The document contains detailed research data on ChatGPT usage patterns, growth statistics, user demographics, and classification of conversation topics. I will now analyze this document and provide insights addressing your three key questions based on the findings presented in this research paper.

Node: write_research_brief

Research Brief Generated:
I need a comprehensive analysis of the NBER Working Paper "How People Use ChatGPT" by Chatterji et al. (2025) that addresses three specific research questions: (1) What are the main findings about


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# Comprehensive Analysis of ChatGPT Usage Patterns: Insights from NBER Research

## Overview of ChatGPT Adoption and Growth

The NBER Working Paper "How People Use ChatGPT" by Chatterji et al. (2025) provides unprecedented insights into the adoption and usage patterns of the world's first mass-market AI chatbot. By July 2025, ChatGPT had achieved remarkable scale, reaching more than 700 million weekly active users—representing approximately 10% of the global adult population [1]. The platform was processing 18 billion messages per week and more than 2.6 billion messages per day, equivalent to over 30,000 messages per second [2].

The speed of adoption has been extraordinary, with no precedent for such rapid global diffusion of a new technology. ChatGPT reached 1 million users within just 5 days of its November 2022 launch and hit 100 million weekly active users in November 2023, less than one year after release [3]. Weekly active users have doubled every 7-8 months since then, with total message volume increasing by 5.8x in the last year alone [3].

## User Demographics and Adoption Patterns

### Gender Distribution Evolution

One of the most striking findings relates to the dramatic shift in gender distribution among ChatGPT users. Early adopters were disproportionately male, with more than 80% of weekly active users having typically male first names when ChatGPT initially launched [1][4]. However, this gender gap has narrowed dramatically over the study period. By July 2025, 52% of active users had typically female first names, suggesting that the gender gap in ChatGPT usage may have closed completely [1][2][4].

### Age Demographics

The user base skews young, with nearly half of all adult messages (46%) coming from users under 26 years old [1]. Interestingly, work usage patterns vary by age, generally increasing with age until users reach 66 and older, who show a drop to 16% work usage [7].

### Geographic Patterns and Global Expansion

The research reveals significant geographic trends in adoption patterns. Higher growth rates have been observed in lower-income countries compared to wealthier nations [2][4]. Geographic disparities are narrowing, with middle-income countries showing 5-6x growth compared to 3x growth in the richest countries [3]. This has resulted in countries like Brazil, South Korea, and the US achieving similar usage rates despite vastly different GDP per capita levels [3]. Growth has been particularly pronounced in low and middle-income countries, with adoption growth there 4× higher than in high-income countries [8].

### Education and Professional Characteristics

Work-related usage is more prevalent among educated users in highly-paid professional occupations [1][2][4]. This demographic concentration is particularly evident in work-related writing tasks, which dominate professional usage patterns across management and business occupations [5].

## Common Use Cases and Conversation Classifications

### Primary Usage Categories

The research employs a sophisticated classification system to categorize ChatGPT conversations. Nearly 80% of all ChatGPT usage falls into three broad categories that collectively define how people interact with AI chatbots [1][2][4]:

**Practical Guidance (29%)**: This represents the most common use case and includes activities such as tutoring and teaching, how-to advice across various topics, creative ideation, and personalized coaching [6][7]. This category demonstrates ChatGPT's role as an advisory tool for decision-making and problem-solving.

**Seeking Information (24%)**: This category includes searching for information about people, current events, products, and recipes, representing conversational search behavior that synthesizes sources [6][7]. This usage pattern appears to be a very close substitute for traditional web search engines but with enhanced conversational capabilities.

**Writing (24%)**: This encompasses automated production of emails, documents, and other communications, but also includes editing, critiquing, summarizing, and translating text provided by users [6][7]. Importantly, about two-thirds of writing requests involve modifying user-provided text rather than creating entirely new content from scratch [1][5].

### Detailed Breakdown of Specific Use Cases

The top 5 specific chat types provide granular insights into user behavior:
1. Seeking specific information - 18.3%
2. Edit provided text - 10.6%
3. Tutoring/teaching - 10.2%
4. How-to advice - 8.5%
5. Personal writing - 8.0% [7]

### Less Common but Notable Categories

Several categories represent smaller but significant portions of usage:

- **Technical Help (7.5%)**: Including programming and data analysis, though computer programming accounts for only 4.2% of all messages, contrary to many expectations about AI usage [1][6][7][8]
- **Multimedia (6.0%)**: Image creation and analysis capabilities [7]
- **Self-Expression (4.3%)**: AI serving as a companion for creative expression [7]
- **Education**: Emerged as a major use case, with 10% of all messages requesting tutoring or teaching [1]
- **Relationships and Personal Reflection**: Account for only 1.9% of messages [5]
- **Therapy/Companionship**: Represents under 1.9% of usage [7]

### Work vs. Non-Work Usage Patterns

A particularly significant finding relates to the evolution of work versus non-work usage. Non-work-related messages have grown dramatically, increasing from 53% in mid-2024 to over 70% of all usage by mid-2025 [1][2][4][6]. While work-related messages showed steady growth, they were significantly outpaced by non-work usage growth rates.

As of June 2025, work-related chats totaled 716 million per day (representing +336% year-over-year growth), while non-work-related chats reached 1,911 million per day (representing +802% year-over-year growth) [7]. This means that only 27% of ChatGPT usage is work-related, with 73% being personal use or what economists classify as "home production" [7].

For work-related usage specifically, writing dominates professional tasks, accounting for 40% of work-related messages in June 2025 [1]. Writing represents by far the most common work use, accounting for 42% of work-related messages overall and more than half of all messages for users in management and business occupations [5].

## Trends and Patterns in Usage Evolution

### User Intent Classification

The researchers introduced an innovative taxonomy to classify user intent, revealing important patterns in how people interact with AI:

- **49% "Asking"**: Seeking information or advice for decision-making purposes [1][7]
- **40% "Doing"**: Requesting specific task performance or completion [1][7]
- **11% "Expressing"**: Social or emotional content and casual conversation [1][7]

For work-related messages, the pattern shifts significantly, with "Doing" increasing to 56%, and writing tasks comprising 35% of all work queries [1]. Notably, "Asking" has grown faster and shows higher user satisfaction rates, indicating ChatGPT's evolution into a decision-support tool rather than just a task automation system [6].

### Growth Trajectory Patterns

The study documented that all user cohorts showed remarkably similar usage patterns: relatively flat usage through most of 2024, followed by substantial increases beginning in early 2025 [3]. This pattern suggests that ChatGPT has improved significantly in terms of capability and/or user-friendliness over the past year [3]. User engagement is intensifying over time, with people sending 40-60% more messages per day than they did initially [3].

### Demographic Shifts Over Time

The most dramatic demographic shift relates to gender distribution, with the user base evolving from 80% male at launch to 52% female by July 2025 [6]. This represents one of the most significant demographic transitions in technology adoption patterns documented in recent research.

The decrease in work-related message share is primarily attributable to changing usage patterns within existing user cohorts rather than changes in the composition of new ChatGPT users. This suggests that as users become more familiar with the technology, they increasingly apply it to personal and non-work contexts.

### Evolution of Usage Types

The data reveals that ChatGPT's role has evolved significantly over the study period. Initially viewed primarily as a work productivity tool, it has increasingly become a general-purpose decision-support and personal assistance platform. The shift toward "Asking" behaviors (up 4% year-over-year to 51.6%) compared to "Doing" behaviors (down 14% share year-over-year to 34.6%) indicates users are increasingly leveraging ChatGPT for advice and decision-making rather than pure task execution [7].

### Economic Value and User Satisfaction

The research provides compelling evidence of ChatGPT's economic value. U.S. users would need to be compensated roughly $98 to give up generative AI for a month, implying at least $97 billion in annual surplus in 2024 alone [6]. User satisfaction remains high, with positive interactions outnumbering negative ones by approximately 4:1 [1].

The economic impact extends beyond traditional productivity metrics to encompass better decision-making, faster learning, and creative ideation—benefits that are often invisible in conventional economic measurements [5]. The authors argue that ChatGPT's strongest economic value lies in serving as a decision-support tool that helps users make choices, think through problems, and produce better writing [1].

## Implications and Future Considerations

The findings suggest that while most economic analysis of AI has focused on productivity impacts in paid work, the impact on activity outside of work represents a similar or potentially larger scale of economic value. The research demonstrates that people are not only outsourcing tasks to AI but are also "insourcing judgment"—using AI to enhance their decision-making capabilities rather than simply replacing human effort [6].

The study's methodology employed strict privacy protections using Data Clean Rooms, automated PII scrubbing, and message classification systems that allowed researchers to analyze patterns without ever viewing actual message content [3]. This privacy-preserving approach, conducted with Harvard IRB approval, provides a model for future research on sensitive user behavior data [4].

### Sources

[1] TECHMANIACS: How People Really Use ChatGPT: Findings from NBER Research: https://techmaniacs.com/2025/09/15/how-people-really-use-chatgpt-findings-from-nber-research/

[2] SSRN: How People Use ChatGPT: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5487080

[3] Forked Lightning by David Deming: How People Use ChatGPT: https://forklightning.substack.com/p/how-people-use-chatgpt

[4] NBER Official Paper: How People Use ChatGPT: https://www.nber.org/papers/w34255

[5] NBER PDF: How People Use ChatGPT: https://www.nber.org/system/files/working_papers/w34255/w34255.pdf

[6] BinaryVerse AI: What Is ChatGPT Used For In 2025, Proven NBER Insights: https://binaryverseai.com/what-is-chatgpt-used-for/

[7] LinkedIn Post by Richard Rosser: ChatGPT use analysis: https://www.linkedin.com/posts/richrosser_chatgpt-activity-7373622180963848192-STWh

[8] LinkedIn Post by Shobhit Sharma: OpenAI and NBER study analysis: https://www.linkedin.com/posts/showbit01_economic-research-chatgpt-usage-paperpdf-activity-7374374572734922752-u_F4


Research workflow completed!


## Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided a PDF and specific questions, it likely proceeded without clarification.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` multiple times to delegate to parallel researchers
- Each delegation specified a focused research topic

### Phase 4: Parallel Research
Multiple researchers worked simultaneously:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive report with:
- Well-structured sections
- Inline citations
- Sources listed at the end
- Balanced coverage of all findings

#### 🏗️ Activity #1: Try Different Configurations

You can experiment with different settings to see how they affect the research.  You may select three or more of the following settings (or invent your own experiments) and describe the results.

### Increase Parallelism
```python
"max_concurrent_research_units": 10  # More researchers working simultaneously
```

### Deeper Research
```python
"max_researcher_iterations": 8   # Supervisor can delegate more times
"max_react_tool_calls": 15      # Each researcher can search more
```

### Use Anthropic Native Search
```python
"search_api": "anthropic"  # Use Claude's built-in web search
```

### Disable Clarification
```python
"allow_clarification": False  # Skip clarification phase
```

ANSWER ✅

Potential Experiments Conducted:

- Increase Parallelism – Allows more researchers to work simultaneously.
- Deeper Research – Increases iterations and tool calls, enabling more thorough exploration of research topics.
- Use Anthropic Native Search – Uses Claude’s built-in web search.
- Disable Clarification – Skips the clarification phase so research starts immediately.

Initial Thoughts:
- Increasing parallelism should yield faster results since more researchers work on concurrent tasks, but it will also increase compute load.
- Deeper research allows for more detailed summaries and better topic exploration, but it may take longer and use more tokens, increasing costs.
- Using Claude’s native search might integrate more smoothly with the model, but results could differ significantly compared to Tavily.
- Skipping clarification can save time, but it increases the risk of misinterpreting the user’s intent.

Trade-offs and Observations:

Increasing parallelism reduced the total runtime (from 6 minutes 22 seconds to 5 minutes 44 seconds) while producing noticeably deeper and more interpretive research reports. The configuration generated richer context and more comprehensive wording, though at the cost of higher token usage, longer generation time, and some redundancy. The lower-parallelism setup produced faster and more concise summaries suitable for quick overviews, whereas the deeper run synthesized findings, highlighted implications, and provided stronger contextual understanding (for example, diving deeper into the economic aspects that the original run missed). Overall, this experiment confirmed the trade-off between speed and analytical depth—greater parallelism improved insight quality but increased computational cost and runtime.

Deeper research also produced more nuanced and comprehensive summaries but significantly increased both token usage and runtime, reinforcing the trade-off between depth and efficiency. When the number of tool calls and iterations was increased, I encountered more rate-limit errors and performance issues. Beyond a certain point, adding more iterations had a diminishing return, providing little additional value while consuming more resources.

Disabling clarification made the process faster since it skipped the step of asking follow-up questions. This also reduced token costs. Ideally, disabling clarification works best when requests are clear and well-defined—especially when sufficient context or structured inputs (such as PDFs) are already provided. The downside is that it may lead to misinterpretations, wasted effort on irrelevant topics, or scoping errors. However, in this case, I did not observe major issues because my prompt was clear, the PDF was supplied beforehand, and I provided three distinct questions. As a result when I ran it, there was little to no noticeable loss in quality.

Overall, these experiments demonstrate that balancing speed, cost, and accuracy is essential. Higher-performance configurations yield deeper and more insightful results but require more resources, careful tuning, and stronger oversight to manage complexity efficiently.




## Key Takeaways

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

## Next Steps

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

### Deploy
- Use LangGraph Cloud for production deployment
- Add proper error handling and logging
- Implement rate limiting and cost controls
- Monitor research quality and costs